In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/passthrough_dataset.csv", parse_dates=["date"])
df = df.set_index("date")
df.head()

,brent,wti,usdmyr,ron95,ron97,diesel,cpi_transport,cpi_headline
date,,,,,,,,
2020-01-31,63.645455,57.519048,4.080110,2.0800,2.5725,2.1800,115.0,122.4
2020-02-29,55.657000,50.542632,4.159900,2.0680,2.3780,2.1340,113.9,122.4
2020-03-31,32.011364,29.207727,4.298409,1.6325,1.9275,1.8150,104.0,120.9
2020-04-30,18.378500,16.547619,4.352486,1.2625,1.5625,1.4675,90.0,117.6
2020-05-31,29.378947,28.562500,4.340906,1.3240,1.6240,1.4740,90.9,117.9


In [4]:
import numpy as np
import pandas as pd
from itertools import product
import statsmodels.api as sm
from statsmodels.tsa.ardl import ARDL, UECM

# df2 already prepared with monthly freq
df2 = df[["diesel", "brent", "usdmyr"]].copy().asfreq("ME")

y = df2["diesel"]
X = df2[["brent", "usdmyr"]]

# --------- brute-force ARDL selection (q can be 0) ----------
best = {"aic": np.inf, "p": None, "q_brent": None, "q_usdmyr": None, "res": None}

for p, q_b, q_u in product(range(1, 5), range(0, 5), range(0, 5)):
    try:
        model = ARDL(y, lags=p, exog=X, order={"brent": q_b, "usdmyr": q_u}, trend="c")
        res = model.fit()
        if res.aic < best["aic"]:
            best.update({"aic": res.aic, "p": p, "q_brent": q_b, "q_usdmyr": q_u, "res": res})
    except Exception:
        continue

print("BEST ARDL by AIC:", {k: best[k] for k in ["aic", "p", "q_brent", "q_usdmyr"]})
print(best["res"].summary())

# --------- fit UECM for ECM representation ----------
p = best["p"]
q_b = best["q_brent"]
q_u = best["q_usdmyr"]

# If your statsmodels UECM refuses q=0, bump only that variable to 1.
try:
    uecm = UECM(y, lags=p, exog=X, order={"brent": q_b, "usdmyr": q_u}, trend="c")
    uecm_res = uecm.fit()
except Exception as e:
    print("UECM failed with q=0 somewhere, bumping q to >=1 where needed. Error:", repr(e))
    uecm = UECM(y, lags=p, exog=X, order={"brent": max(1, q_b), "usdmyr": max(1, q_u)}, trend="c")
    uecm_res = uecm.fit()

print("\nUECM (ECM form) summary:")
print(uecm_res.summary())


BEST ARDL by AIC: {'aic': np.float64(-75.58044334017673), 'p': 4, 'q_brent': 0, 'q_usdmyr': 0}
                              ARDL Model Results                              
Dep. Variable:                 diesel   No. Observations:                   71
Model:                  ARDL(4, 0, 0)   Log Likelihood                  45.790
Method:               Conditional MLE   S.D. of innovations              0.122
Date:                Thu, 08 Jan 2026   AIC                            -75.580
Time:                        04:20:46   BIC                            -57.943
Sample:                    05-31-2020   HQIC                           -68.601
                         - 11-30-2025                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1890      0.348     -0.543      0.589      -0.886       0.508
diesel.L1      1.2260      0.123    

In [5]:
df_policy = df2.copy()

df_policy["diesel_subsidy_removed"] = (
    df_policy.index >= "2024-06-01"
).astype(int)


In [6]:
from statsmodels.tsa.ardl import ARDL
from itertools import product
import numpy as np

y = df_policy["diesel"]
X = df_policy[["brent", "usdmyr", "diesel_subsidy_removed"]]

best = {"aic": np.inf}

for p, q_b, q_u in product(range(1, 5), range(0, 5), range(0, 5)):
    try:
        model = ARDL(
            y,
            lags=p,
            exog=X,
            order={
                "brent": q_b,
                "usdmyr": q_u,
                "diesel_subsidy_removed": 0
            },
            trend="c"
        )
        res = model.fit()
        if res.aic < best["aic"]:
            best = {
                "aic": res.aic,
                "p": p,
                "q_brent": q_b,
                "q_usdmyr": q_u,
                "res": res
            }
    except Exception:
        continue

print("BEST MODEL WITH POLICY DUMMY:", best)
print(best["res"].summary())


BEST MODEL WITH POLICY DUMMY: {'aic': np.float64(-147.04527170932394), 'p': 2, 'q_brent': 0, 'q_usdmyr': 0, 'res': <statsmodels.tsa.ardl.model.ARDLResultsWrapper object at 0x0000020F0607E930>}
                              ARDL Model Results                              
Dep. Variable:                 diesel   No. Observations:                   71
Model:               ARDL(2, 0, 0, 0)   Log Likelihood                  80.523
Method:               Conditional MLE   S.D. of innovations              0.075
Date:                Thu, 08 Jan 2026   AIC                           -147.045
Time:                        04:20:53   BIC                           -131.407
Sample:                    03-31-2020   HQIC                          -140.841
                         - 11-30-2025                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------

In [7]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

tmp = df2[["brent","usdmyr"]].dropna()
print("Correlation brent vs usdmyr:", tmp["brent"].corr(tmp["usdmyr"]))

X = sm.add_constant(tmp)
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns
)
print(vif)


Correlation brent vs usdmyr: 0.4232663144540387
const     462.909877
brent       1.218256
usdmyr      1.218256
dtype: float64


Residual diagnostics for ARDL validity

In [16]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat

def bg_test_ardl_aligned(res, nlags=4):
    """
    Breusch–Godfrey for ARDL results with robust alignment when resid length < exog rows.
    Aligns exog to residual sample by taking last len(resid) rows.
    """
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    n_x = X0_full.shape[0]

    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}. Try nlags=1 or 2.")

    # Align exog to residual sample (take last n_u rows)
    X0 = X0_full[-n_u:, :]

    # Build lagged residuals (trimmed)
    U_lags = lagmat(u, maxlag=nlags, trim='both')  # shape: (n_u - nlags, nlags)
    y = u[nlags:]                                  # shape: (n_u - nlags,)

    X = np.column_stack([X0[nlags:, :], U_lags])

    # Safety checks
    if not np.isfinite(X).all():
        raise ValueError("Non-finite values (NaN/inf) found in X after alignment.")
    if not np.isfinite(y).all():
        raise ValueError("Non-finite values (NaN/inf) found in y after alignment.")

    aux_res = sm.OLS(y, X).fit()

    # LM = n * R^2
    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    # Joint F-test on lagged residuals
    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(n_x)
    }


In [17]:
bg_test_ardl_aligned(res, nlags=4)


{'LM stat': 20.252887089039188,
 'LM p-value': 0.00044514122434169645,
 'F stat': 5.407562906005031,
 'F p-value': 0.0009396430922173654,
 'nobs_aux': 63,
 'resid_len': 67,
 'exog_rows_full': 71}

In [18]:
bg_test_ardl_aligned(res, nlags=2)


{'LM stat': 18.373155246451393,
 'LM p-value': 0.00010240473325582631,
 'F stat': 11.528090751548607,
 'F p-value': 5.799341634766646e-05,
 'nobs_aux': 65,
 'resid_len': 67,
 'exog_rows_full': 71}

In [20]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

u = np.asarray(res.resid)

X0_full = np.asarray(res.model.data.exog)
X0 = X0_full[-len(u):, :]          # align to residual sample
X0 = sm.add_constant(X0, has_constant='add')   # <-- force constant

bp = het_breuschpagan(u, X0)

bp_out = {
    "LM stat": float(bp[0]),
    "LM p-value": float(bp[1]),
    "F stat": float(bp[2]),
    "F p-value": float(bp[3]),
    "k_exog": int(X0.shape[1])
}

bp_out


{'LM stat': 2.4744754097045707,
 'LM p-value': 0.4799216142542523,
 'F stat': 0.8053244655311369,
 'F p-value': 0.4956046466826779,
 'k_exog': 4}

In [21]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import joblib

from scipy import stats
from statsmodels.tsa.tsatools import lagmat
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.diagnostic import recursive_olsresiduals


In [22]:
def bg_test_ardl_aligned(res, nlags=2):
    """
    Breusch–Godfrey for ARDL results with robust alignment when resid length < exog rows.
    Aligns exog to residual sample by taking last len(resid) rows.
    """
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    n_x = X0_full.shape[0]

    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}. Try nlags=1.")

    # Align exog to residual sample
    X0 = X0_full[-n_u:, :]

    # Lagged residuals + trim
    U_lags = lagmat(u, maxlag=nlags, trim='both')  # (n_u - nlags, nlags)
    y = u[nlags:]                                  # (n_u - nlags,)
    X = np.column_stack([X0[nlags:, :], U_lags])

    if not np.isfinite(X).all():
        raise ValueError("Non-finite values (NaN/inf) found in X after alignment.")
    if not np.isfinite(y).all():
        raise ValueError("Non-finite values (NaN/inf) found in y after alignment.")

    aux_res = sm.OLS(y, X).fit()

    # LM = n * R^2
    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    # Joint F-test on lagged residuals
    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(n_x)
    }


In [ ]:
bg_out = bg_test_ardl_aligned(res, nlags=2)  # use 2 for your sample size
bg_out


{'LM stat': 18.373155246451393,
 'LM p-value': 0.00010240473325582631,
 'F stat': 11.528090751548607,
 'F p-value': 5.799341634766646e-05,
 'nobs_aux': 65,
 'resid_len': 67,
 'exog_rows_full': 71}

In [24]:
def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]  # align
    X0 = sm.add_constant(X0, has_constant="add")  # force constant

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
        "k_exog": int(X0.shape[1])
    }

bp_out = bp_test_ardl(res)
bp_out


{'LM stat': 2.4744754097045707,
 'LM p-value': 0.4799216142542523,
 'F stat': 0.8053244655311369,
 'F p-value': 0.4956046466826779,
 'k_exog': 4}

In [28]:
import joblib
joblib.dump(res, "../data/joblib/ardl_diesel_brent.joblib")

['../data/joblib/ardl_diesel_brent.joblib']